# Exploratory Data Analysis: NIH Malaria Cell Images Dataset
## Microscopic Diagnosis of Plasmodium Falciparum via Deep Learning

**Author:** Senior Machine Learning Engineer & Deep Learning Researcher  
**Dataset:** NIH Malaria Cell Images (27,558 Giemsa-stained thin blood smear cell images)

---

### 1. Clinical Context & Dataset Overview
Malaria is a life-threatening disease caused by *Plasmodium* parasites transmitted through the bites of infected female *Anopheles* mosquitoes. The gold standard for malaria diagnosis remains the microscopic examination of Giemsa-stained thin blood smears. Segmented red blood cells (erythrocytes) are classified into:
1. **Parasitized:** Cells containing intracellular malaria parasites (distinct chromatic ring forms, trophozoites, or schizonts).
2. **Uninfected:** Normal, healthy erythrocytes showing no intracellular inclusions.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["font.sans-serif"] = "Arial"

DATA_DIR = "../data"
MANIFEST_PATH = os.path.join(DATA_DIR, "dataset_manifest.csv")

df = pd.read_csv(MANIFEST_PATH)
print(f"Total Images Loaded: {len(df):,}")
df.head()

### 2. Class Distribution and Balance
The dataset contains exactly 27,558 cell images. Let's inspect class frequencies and balance.

In [ ]:
class_counts = df['category'].value_counts()
print("Class Distribution:")
print(class_counts)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=["#d62728", "#1f77b4"], ax=ax)
ax.set_title("Class Distribution (Perfect 50/50 Balance)", weight="bold")
ax.set_ylabel("Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width()/2., p.get_height()/2),
                ha='center', va='center', color='white', weight='bold')
plt.show()

### 3. Patient Grouping and Data Leakage Analysis
A critical scientific requirement is avoiding patient-level data leakage. The filenames encode slide/patient IDs:
- `C100P61...`: Slide 100 from Patient 61 (Infected individual)
- `C12N...`: Slide 12 from Control donor (Uninfected individual)

In [ ]:
patient_counts = df['patient_id'].nunique()
print(f"Total Distinct Patients/Slide Donors: {patient_counts}")

split_summary = df.groupby(['split', 'category']).size().unstack()
print("\nDataset Partition by Class:")
display(split_summary)

# Verification of zero patient overlap
train_p = set(df[df['split'] == 'train']['patient_id'])
val_p = set(df[df['split'] == 'val']['patient_id'])
test_p = set(df[df['split'] == 'test']['patient_id'])

print(f"Patient overlap Train/Val: {len(train_p & val_p)}")
print(f"Patient overlap Train/Test: {len(train_p & test_p)}")
print(f"Patient overlap Val/Test: {len(val_p & test_p)}")

### 4. Microscopic Cell Image Inspection
Let's visually contrast representative Parasitized cells vs Uninfected erythrocytes.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
parasitized_samples = df[df['category'] == 'Parasitized'].sample(4, random_state=42)
uninfected_samples = df[df['category'] == 'Uninfected'].sample(4, random_state=42)

for col, (_, row) in enumerate(parasitized_samples.iterrows()):
    img = Image.open(row['filepath'])
    axes[0, col].imshow(img)
    axes[0, col].set_title(f"Parasitized ({row['patient_id']})\nDim: {img.size[0]}x{img.size[1]}", color="#d62728", weight="bold")
    axes[0, col].axis('off')

for col, (_, row) in enumerate(uninfected_samples.iterrows()):
    img = Image.open(row['filepath'])
    axes[1, col].imshow(img)
    axes[1, col].set_title(f"Uninfected ({row['patient_id']})\nDim: {img.size[0]}x{img.size[1]}", color="#1f77b4", weight="bold")
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

### 5. Image Dimension and Channel Analysis
Microscopy images have variable pixel resolutions due to cell crop boundaries. We analyze image dimensions to validate our choice of **128 × 128** resizing.

In [ ]:
sample_df = df.sample(1000, random_state=42)
dims = [Image.open(p).size for p in sample_df['filepath']]
widths, heights = zip(*dims)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(widths, kde=True, ax=ax1, color="#4c72b0")
ax1.axvline(128, color="red", linestyle="--", label="Target Resize (128x128)")
ax1.set_title("Distribution of Cell Image Widths (Mean: ~132px)", weight="bold")
ax1.legend()

sns.histplot(heights, kde=True, ax=ax2, color="#55a868")
ax2.axvline(128, color="red", linestyle="--", label="Target Resize (128x128)")
ax2.set_title("Distribution of Cell Image Heights (Mean: ~133px)", weight="bold")
ax2.legend()

plt.tight_layout()
plt.show()

### 6. Conclusions & Modeling Strategy
1. **Zero Patient Overlap:** Strict patient-level grouping is enforced across train, val, and test partitions.
2. **Dimensionality:** Resizing to 128 × 128 preserves morphological parasite characteristics while maintaining high computational efficiency.
3. **Regularization:** Given the high resolution and slide-specific stain variations, multi-tier regularization (Spatial Dropout, BatchNorm, L2, Global Average Pooling, and conservative data augmentation) is essential to ensure generalizability.